# Sequences Before Attention: RNNs, Vanishing Gradients, and Why Transformers Won

The full code of [Sequences Before Attention: RNNs, Vanishing Gradients, and Why Transformers Won](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/rnns-and-why-transformers-won) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rnns-and-why-transformers-won.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 215 min end to end on the RTX 3090 machine that saved the outputs below.
Trains about thirty small recurrent models; the longest part of the series' Part II to rerun. At 40 digits the attention model's training is unstable, so whether a seed trains varies by machine.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.11.0+cu130, 214 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.11.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "rnns-and-why-transformers-won"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/rnns-and-why-transformers-won


## `bptt.py`

An Elman RNN with a 2-number state run for 3 steps, then backpropagation through time by hand, checked against finite differences.

In [2]:
!{sys.executable} bptt.py

h_1 = [ 0.537  -0.2913]
h_2 = [ 0.5691 -0.0673]
h_3 = [0.082 0.357]
prediction y = h_3 . w_out = -0.2750
gradient reaching h_2: [ 0.6201 -0.9208]
gradient reaching h_1: [ 0.1521 -0.7177]
gradient reaching h_0: [-0.0448 -0.4374]
dy/dW_h = [[0.7904, -0.9888], [-0.189, 0.3257]]; largest gap to finite differences 5.1e-11


The post printed:

```text
h_1 = [ 0.537  -0.2913]
h_2 = [ 0.5691 -0.0673]
h_3 = [0.082 0.357]
prediction y = h_3 . w_out = -0.2750
gradient reaching h_2: [ 0.6201 -0.9208]
gradient reaching h_1: [ 0.1521 -0.7177]
gradient reaching h_0: [-0.0448 -0.4374]
dy/dW_h = [[0.7904, -0.9888], [-0.189, 0.3257]]; largest gap to finite differences 4.6e-11
```

## `vanishing.py`

How much of a gradient survives 1, 10, 25 and 50 steps back through a 64-wide recurrence, by the recurrent matrix's scale, linear and with tanh.

In [3]:
!{sys.executable} vanishing.py

radius 0.9, linear: gradient size after 1 steps 9.0e-01, 10 steps 3.5e-01, 25 steps 7.2e-02, 50 steps 5.2e-03
radius 0.9, tanh  : gradient size after 1 steps 6.6e-01, 10 steps 1.6e-02, 25 steps 1.2e-04, 50 steps 1.6e-08
radius 1.0, linear: gradient size after 1 steps 1.0e+00, 10 steps 1.0e+00, 25 steps 1.0e+00, 50 steps 1.0e+00
radius 1.0, tanh  : gradient size after 1 steps 7.4e-01, 10 steps 6.1e-02, 25 steps 1.1e-03, 50 steps 1.8e-06
radius 1.1, linear: gradient size after 1 steps 1.1e+00, 10 steps 2.6e+00, 25 steps 1.1e+01, 50 steps 1.2e+02
radius 1.1, tanh  : gradient size after 1 steps 8.4e-01, 10 steps 1.8e-01, 25 steps 5.3e-03, 50 steps 2.8e-05
radius 2.0, tanh  : gradient size after 1 steps 1.1e+00, 10 steps 2.9e+00, 25 steps 1.3e+01, 50 steps 1.1e+02
radius 1.1, tanh, no inputs: gradient size after 1 steps 1.0e+00, 10 steps 9.8e-01, 25 steps 1.0e+00, 50 steps 8.7e-01


The post printed:

```text
radius 0.9, linear: gradient size after 1 steps 9.0e-01, 10 steps 3.5e-01, 25 steps 7.2e-02, 50 steps 5.2e-03
radius 0.9, tanh  : gradient size after 1 steps 6.6e-01, 10 steps 1.6e-02, 25 steps 1.2e-04, 50 steps 1.6e-08
radius 1.0, linear: gradient size after 1 steps 1.0e+00, 10 steps 1.0e+00, 25 steps 1.0e+00, 50 steps 1.0e+00
radius 1.0, tanh  : gradient size after 1 steps 7.4e-01, 10 steps 6.1e-02, 25 steps 1.1e-03, 50 steps 1.8e-06
radius 1.1, linear: gradient size after 1 steps 1.1e+00, 10 steps 2.6e+00, 25 steps 1.1e+01, 50 steps 1.2e+02
radius 1.1, tanh  : gradient size after 1 steps 8.4e-01, 10 steps 1.8e-01, 25 steps 5.3e-03, 50 steps 2.8e-05
radius 2.0, tanh  : gradient size after 1 steps 1.1e+00, 10 steps 2.9e+00, 25 steps 1.3e+01, 50 steps 1.1e+02
radius 1.1, tanh, no inputs: gradient size after 1 steps 1.0e+00, 10 steps 9.8e-01, 25 steps 1.0e+00, 50 steps 8.7e-01
```

## `gates.py`

The gradient reaching the input 1 to 50 steps back in untrained RNN, GRU and LSTM cells, and how the GRU's update bias and the LSTM's forget bias change it.

In [4]:
!{sys.executable} gates.py

RNN                 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 3.5e+00 / 2.4e-03 / 1.4e-08 / 3.5e-17


GRU, default        : gradient reaching the input 1 / 10 / 25 / 50 steps back: 2.2e+00 / 1.6e-02 / 1.3e-05 / 6.2e-11
GRU, update bias +1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.3e+00 / 9.9e-02 / 1.7e-03 / 3.5e-06


LSTM, forget bias 0 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 6.7e-03 / 2.1e-06 / 8.1e-12
LSTM, forget bias 1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 8.6e-02 / 3.1e-03 / 1.8e-05
LSTM, forget bias 3 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 8.9e-01 / 4.1e-01 / 4.3e-01 / 4.6e-01


The post printed:

```text
RNN                 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 3.5e+00 / 2.4e-03 / 1.4e-08 / 3.5e-17
GRU, default        : gradient reaching the input 1 / 10 / 25 / 50 steps back: 2.2e+00 / 1.6e-02 / 1.3e-05 / 6.2e-11
GRU, update bias +1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.3e+00 / 9.9e-02 / 1.7e-03 / 3.5e-06
LSTM, forget bias 0 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 6.7e-03 / 2.1e-06 / 8.1e-12
LSTM, forget bias 1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 8.6e-02 / 3.1e-03 / 1.8e-05
LSTM, forget bias 3 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 8.9e-01 / 4.1e-01 / 4.3e-01 / 4.6e-01
```

## `corpus.py`

The training and validation text: the blog's English posts as of 2026-09-30, two posts held out. Reports their size and character set.

In [5]:
!{sys.executable} corpus.py

train 593,616 characters, validation 56,786; 147 distinct characters


validation characters never seen in training: 123 of 56,786


The post printed:

```text
train 593,616 characters, validation 56,786; 147 distinct characters
validation characters never seen in training: 123 of 56,786
```

## `ngram.py`

Count models of the previous 0 to 4 characters with add-one smoothing, scored on the held-out posts.

In [6]:
!{sys.executable} ngram.py

previous 0 characters: validation 3.468 nats/char = 5.003 bits/char


previous 1 characters: validation 2.849 nats/char = 4.110 bits/char


previous 2 characters: validation 2.481 nats/char = 3.580 bits/char


previous 3 characters: validation 2.628 nats/char = 3.791 bits/char


previous 4 characters: validation 3.112 nats/char = 4.490 bits/char


The post printed:

```text
previous 0 characters: validation 3.468 nats/char = 5.003 bits/char
previous 1 characters: validation 2.849 nats/char = 4.110 bits/char
previous 2 characters: validation 2.481 nats/char = 3.580 bits/char
previous 3 characters: validation 2.628 nats/char = 3.791 bits/char
previous 4 characters: validation 3.112 nats/char = 4.490 bits/char
```

## `kneser_ney.py`

Interpolated Kneser–Ney count models of the previous 1 to 7 characters, and a check that their probabilities sum to one.

In [7]:
!{sys.executable} kneser_ney.py

previous 1 characters: 4.124 bits/char


previous 2 characters: 3.370 bits/char


previous 3 characters: 2.844 bits/char


previous 4 characters: 2.650 bits/char


previous 5 characters: 2.600 bits/char


previous 6 characters: 2.594 bits/char


previous 7 characters: 2.606 bits/char


probabilities over all characters sum to 1.000000 after 'the'


The post printed:

```text
previous 1 characters: 4.124 bits/char
previous 2 characters: 3.370 bits/char
previous 3 characters: 2.844 bits/char
previous 4 characters: 2.650 bits/char
previous 5 characters: 2.600 bits/char
previous 6 characters: 2.594 bits/char
previous 7 characters: 2.606 bits/char
probabilities over all characters sum to 1.000000 after 'the'
```

## `char_models.py`

Runs `charlm.py` for each cell with seeds 0, 1 and 2, at 3,000 steps.

In [8]:
!{sys.executable} char_models.py

RNN  seed 0, 3,000 steps: 129,940 parameters, 192 s; train 1.941, validation 2.839 bits/char


RNN  seed 1, 3,000 steps: 129,940 parameters, 199 s; train 1.945, validation 2.829 bits/char


RNN  seed 2, 3,000 steps: 129,940 parameters, 187 s; train 1.947, validation 2.831 bits/char


GRU  seed 0, 3,000 steps: 294,804 parameters, 383 s; train 1.549, validation 2.828 bits/char


GRU  seed 1, 3,000 steps: 294,804 parameters, 412 s; train 1.576, validation 2.849 bits/char


GRU  seed 2, 3,000 steps: 294,804 parameters, 499 s; train 1.622, validation 2.819 bits/char


LSTM seed 0, 3,000 steps: 377,236 parameters, 382 s; train 1.714, validation 2.694 bits/char


LSTM seed 1, 3,000 steps: 377,236 parameters, 363 s; train 1.729, validation 2.695 bits/char


LSTM seed 2, 3,000 steps: 377,236 parameters, 401 s; train 1.777, validation 2.683 bits/char


The post printed:

```text
RNN  seed 0, 3,000 steps: 129,940 parameters, 58 s; train 1.941, validation 2.839 bits/char
RNN  seed 1, 3,000 steps: 129,940 parameters, 52 s; train 1.946, validation 2.829 bits/char
RNN  seed 2, 3,000 steps: 129,940 parameters, 52 s; train 1.947, validation 2.832 bits/char
GRU  seed 0, 3,000 steps: 294,804 parameters, 141 s; train 1.547, validation 2.827 bits/char
GRU  seed 1, 3,000 steps: 294,804 parameters, 173 s; train 1.575, validation 2.842 bits/char
GRU  seed 2, 3,000 steps: 294,804 parameters, 168 s; train 1.615, validation 2.812 bits/char
LSTM seed 0, 3,000 steps: 377,236 parameters, 263 s; train 1.714, validation 2.694 bits/char
LSTM seed 1, 3,000 steps: 377,236 parameters, 268 s; train 1.729, validation 2.696 bits/char
LSTM seed 2, 3,000 steps: 377,236 parameters, 257 s; train 1.781, validation 2.690 bits/char
```

## `char_models_long.py`

Runs `charlm.py` for each cell with seed 0 at 12,000 steps.

In [9]:
!{sys.executable} char_models_long.py

RNN  seed 0, 12,000 steps: 129,940 parameters, 913 s; train 1.789, validation 3.008 bits/char


GRU  seed 0, 12,000 steps: 294,804 parameters, 1621 s; train 1.345, validation 3.175 bits/char


LSTM seed 0, 12,000 steps: 377,236 parameters, 1380 s; train 1.200, validation 3.355 bits/char


The post printed:

```text
RNN  seed 0, 12,000 steps: 129,940 parameters, 326 s; train 1.793, validation 3.017 bits/char
GRU  seed 0, 12,000 steps: 294,804 parameters, 737 s; train 1.351, validation 3.182 bits/char
LSTM seed 0, 12,000 steps: 377,236 parameters, 761 s; train 1.211, validation 3.369 bits/char
```

## `reverse.py`

Runs `seq2seq.py` at lengths 5, 10, 20 and 40, with and without attention, seeds 0 and 1, at 2,000 steps.

In [10]:
!{sys.executable} reverse.py

length  5, no attention, seed 0: digits right 0.987, whole sequences right 0.967


length  5, no attention, seed 1: digits right 0.995, whole sequences right 0.986


length  5, attention   , seed 0: digits right 0.987, whole sequences right 0.965


length  5, attention   , seed 1: digits right 0.998, whole sequences right 0.994


length 10, no attention, seed 0: digits right 0.974, whole sequences right 0.883


...

length 20, attention   , seed 1: digits right 0.952, whole sequences right 0.874


length 40, no attention, seed 0: digits right 0.400, whole sequences right 0.000


length 40, no attention, seed 1: digits right 0.386, whole sequences right 0.000


length 40, attention   , seed 0: digits right 0.345, whole sequences right 0.007


length 40, attention   , seed 1: digits right 0.741, whole sequences right 0.466


The post printed:

```text
length  5, no attention, seed 0: digits right 0.990, whole sequences right 0.972
length  5, no attention, seed 1: digits right 0.994, whole sequences right 0.979
length  5, attention   , seed 0: digits right 0.998, whole sequences right 0.996
length  5, attention   , seed 1: digits right 1.000, whole sequences right 1.000
length 10, no attention, seed 0: digits right 0.962, whole sequences right 0.832
length 10, no attention, seed 1: digits right 0.971, whole sequences right 0.853
length 10, attention   , seed 0: digits right 0.987, whole sequences right 0.958
length 10, attention   , seed 1: digits right 0.984, whole sequences right 0.958
length 20, no attention, seed 0: digits right 0.784, whole sequences right 0.148
length 20, no attention, seed 1: digits right 0.818, whole sequences right 0.189
length 20, attention   , seed 0: digits right 0.938, whole sequences right 0.840
length 20, attention   , seed 1: digits right 0.944, whole sequences right 0.869
length 40, no attention, seed 0: digits right 0.374, whole sequences right 0.000
length 40, no attention, seed 1: digits right 0.379, whole sequences right 0.000
length 40, attention   , seed 0: digits right 0.479, whole sequences right 0.106
length 40, attention   , seed 1: digits right 0.250, whole sequences right 0.000
```

## `reverse_40_longer.py`

Length 40, both models, 3,000 steps at the original learning rate.

In [11]:
!{sys.executable} reverse_40_longer.py

length 40, no attention, seed 0, 3,000 steps: digits right 0.477, whole sequences right 0.000


length 40, no attention, seed 1, 3,000 steps: digits right 0.501, whole sequences right 0.000


length 40, attention   , seed 0, 3,000 steps: digits right 0.176, whole sequences right 0.000


length 40, attention   , seed 1, 3,000 steps: digits right 0.172, whole sequences right 0.000


The post printed:

```text
length 40, no attention, seed 0, 3,000 steps: digits right 0.490, whole sequences right 0.000
length 40, no attention, seed 1, 3,000 steps: digits right 0.513, whole sequences right 0.000
length 40, attention   , seed 0, 3,000 steps: digits right 0.195, whole sequences right 0.000
length 40, attention   , seed 1, 3,000 steps: digits right 0.177, whole sequences right 0.000
```

## `reverse_40_lr.py`

Length 40, both models, 3,000 steps at learning rate 1e-3.

In [12]:
!{sys.executable} reverse_40_lr.py

length 40, no attention, seed 0, 3,000 steps, learning rate 0.001: digits right 0.230, whole sequences right 0.000


length 40, no attention, seed 1, 3,000 steps, learning rate 0.001: digits right 0.234, whole sequences right 0.000


length 40, attention   , seed 0, 3,000 steps, learning rate 0.001: digits right 0.975, whole sequences right 0.937


length 40, attention   , seed 1, 3,000 steps, learning rate 0.001: digits right 0.976, whole sequences right 0.952


The post printed:

```text
length 40, no attention, seed 0, 3,000 steps, learning rate 0.001: digits right 0.224, whole sequences right 0.000
length 40, no attention, seed 1, 3,000 steps, learning rate 0.001: digits right 0.238, whole sequences right 0.000
length 40, attention   , seed 0, 3,000 steps, learning rate 0.001: digits right 0.971, whole sequences right 0.927
length 40, attention   , seed 1, 3,000 steps, learning rate 0.001: digits right 0.975, whole sequences right 0.940
```

## `reverse_40_6000.py`

Length 40, without attention, 6,000 steps.

In [13]:
!{sys.executable} reverse_40_6000.py

length 40, no attention, seed 0, 6,000 steps: digits right 0.715, whole sequences right 0.016


length 40, no attention, seed 1, 6,000 steps: digits right 0.719, whole sequences right 0.007


The post printed:

```text
length 40, no attention, seed 0, 6,000 steps: digits right 0.701, whole sequences right 0.006
length 40, no attention, seed 1, 6,000 steps: digits right 0.745, whole sequences right 0.015
```

## `parallel.py`

One 1,024-token sequence through a GRU (1,024 sequential steps) and through causal self-attention (one pass): FLOPs and time.

In [14]:
!{sys.executable} parallel.py

GRU:       1024 sequential steps, 0.81 GFLOPs,   42.1 ms
attention: 1 parallel pass,      1.48 GFLOPs,    9.7 ms


The post printed:

```text
GRU:       1024 sequential steps, 0.81 GFLOPs,    9.5 ms
attention: 1 parallel pass,      1.48 GFLOPs,    1.5 ms
```